In [ ]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Piecewise costs and constraints

In this example, we demonstrate how to represent non-linear system relationships as piecewise-linear curves in PyPSA. 
This allows us to describe such relationships as:

- Economies of scale (investment costs per unit capacity reduce as an asset's capacity increases)
- Marginal cost offer curves (marginal cost per unit dispatched energy changes an asset's dispatch increases)
- Part-load efficiencies (efficiency varies when an asset operates below its rated capacity)

Here, we build a model from the [single-node capacity expansion example]() and add a combined-cycle gas turbine with piecewise curves for each of the above relationships. 
Note that the goal here is not necessarily to showcase a realistic scenario, but rather to demonstrate how it is possible to model piecewise costs and constraints in PyPSA.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
n = pypsa.examples.model_energy()
n.remove("Generator", "load shedding")

## Reference case

Our reference case will be a model in which the rated values for our CCGT are used.

In [ ]:
def create_new_network(n: pypsa.Network, **ccgt_kwargs) -> pypsa.Network:
    n_new = n.copy()

    n_new.add("Generator", "CCGT", carrier="CCGT", bus="electricity", **ccgt_kwargs)
    n_new.add("Carrier", "CCGT", co2_emissions=0.2)

    n_new.add(
        "GlobalConstraint",
        "co2_limit",
        type="primary_energy",
        carrier_attribute="co2_emissions",
        sense="<=",
        constant=1e4,
    )
    return n_new


n_ref = create_new_network(
    n,
    p_nom_extendable=True,
    p_nom_max=1000,
    capital_cost=150000,
    marginal_cost=50,
    efficiency=0.58,
)
n_ref.optimize()

## Creating piecewise relationships

Let's assume the following relationships:

1. The marginal investment cost of our CCGT plant decreases until a tipping point where additional capacity faces a *diseconomy*.
   This could be caused by land purchase costs increasing for additional capacity beyond a certain level.
   We will describe this with a sigmoid-like curve:

   $$
   \frac{C_{inv}}{C_{inv}^{max}}\ =\frac{\left(\ (\frac{P_{nom}}{P_{nom}^{max}}-1/2)\sqrt{\left(1-k/4\right)}\right)}{\sqrt{\left(1-k(\frac{P_{nom}}{P_{nom}^{max}}-1/2)^{2}\right)}}+1/2
   $$

   For this example, we will use a scaling factor $k$ of 2.4

2. The efficiency of a CCGT degrades at part-load.
   Let's assume it follows a logarithmic curve bounded by $\eta$ = 58% at full load and $\eta$ = 45% at 20% load:

   $$
   \eta = \eta^{max} + 0.0808 \cdot \ln(P/P^{max})
   $$

3. The CCGT asset in PyPSA represents a collection of real plants, each with different marginal costs which will be applied in order of merit. 
   We will represent this marginal cost offer curve as:

   $$
   C_m = C_0 + \alpha \cdot \frac{P}{P^{max}} + \beta \cdot (\frac{P}{P^{max}})^3
   $$

   For this example, let's assume $C_0$ = 35, $\alpha$ = 15 €/MWh, $\beta$ = 30€/MW³h

We can approximate these non-linear relationships with a series of straight lines or `segments`. 
Here, we will heuristically select points along each curve to derive these segments.

In [ ]:
def plot_piecewise_constraints(
    x_max, n_breakpoints, y_ref, y_func, title, xlabel, ylabel, min_x=0
):
    x_continuous = np.linspace(min_x, x_max, 500)
    x_breakpoints = np.linspace(min_x, x_max, n_breakpoints)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(x_continuous, y_func(x_continuous), lw=2, label="Non-linear curve")
    ax.plot(x_breakpoints, y_func(x_breakpoints), "--o", lw=2, label="Piecewise linear")
    ax.plot([min_x, x_max], y_ref, "-.o", lw=2, label="Reference linear")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 1. Investment cost – economy of scale followed by diseconomy of scale
P_max_inv = 1000  # MW
C_max_inv = 1500000 * P_max_inv  # €
k = 3


def get_inv_cost_curve(P: np.array):
    p = P / P_max_inv - 0.5
    return C_max_inv * (p * np.sqrt(1 - k / 4) / np.sqrt(1 - k * p**2) + 0.5)


plot_piecewise_constraints(
    P_max_inv,
    4,
    (0, C_max_inv),
    get_inv_cost_curve,
    "Investment cost piecewise curve approximation",
    "Capacity (MW)",
    "Investment cost (€)",
)

In [ ]:
# 2. Part-load efficiency
eta_rated = 0.58  # target efficiency at rated load
P_frac_min = 0.2  # minimum load fraction to avoid log(0) and very low efficiency


def get_part_load_eff_curve(P_frac):
    return 0.0808 * np.log(P_frac) + eta_rated


plot_piecewise_constraints(
    1,
    4,
    (eta_rated, eta_rated),
    get_part_load_eff_curve,
    "Part-load efficiency",
    "Load rate (Dispatch / Rated dispatch)",
    "Efficiency",
    min_x=P_frac_min,
)

In [ ]:
# 3. Marginal cost offer curve
C0_mc = 35  # €/h
alpha_mc = 15.0  # €/MWh
beta_mc = 30  # €/MW²h


def get_marginal_cost_curve(P_frac):
    return C0_mc + alpha_mc * P_frac + beta_mc * P_frac**3


ref_mc = C0_mc + alpha_mc + beta_mc
plot_piecewise_constraints(
    1,
    4,
    (ref_mc, ref_mc),
    get_marginal_cost_curve,
    "Marginal cost offer curve",
    "Load rate (Dispatch / Rated dispatch)",
    "Cost (€/MWh)",
)

## Piecewise attributes for extendable components

Of the above curves, only (dis)economies of scale can be applied to the case where our CCGT component is extendable.
The other two relate to the load rate of a component (`p/p_nom`) which is a non-linear relationship of two decision variables when the component is extendable.

Let's see what happens when we add this piecewise constraint:

In [ ]:
p_nom_breakpoints = np.array([0, 250, 500, 750, 1000])
capital_costs = get_inv_cost_curve(p_nom_breakpoints)
n_piecewise_inv = create_new_network(
    n,
    p_nom_extendable=True,
    p_nom_max=1000,
    capital_cost=dict(zip(p_nom_breakpoints, capital_costs)),
    marginal_cost=50,
    efficiency=0.58,
)
n_piecewise_inv.optimize()

We can see the results stored in a new component `segments` data table:

In [ ]:
display(n_piecewise_inv.c.generators.segments.capital_cost)

## Piecewise attributes for non-extendable components

Let's fix the capacity and apply our marginal cost and efficiency piecewise constraints.

In [ ]:
p_pu_breakpoints = np.array([0.2, 0.4, 0.6, 0.8, 1.0])
efficiencies = get_part_load_eff_curve(p_pu_breakpoints)
marginal_cost = get_marginal_cost_curve(p_pu_breakpoints)

n_piecewise_dispatch = create_new_network(
    n,
    p_nom_extendable=False,
    p_nom=1000,
    capital_cost=150000,
    p_min_pu=0.2,
    efficiency=dict(zip(p_pu_breakpoints, efficiencies)),
    marginal_cost=dict(zip(p_pu_breakpoints, marginal_cost)),
)
n_piecewise_dispatch.optimize()